# UMC: Taiwan ordinary (2303.TW) vs US ADR (UMC)

Question: after normalizing the ADR by its conversion ratio and FX, do the two
trade as the same asset?

**Facts used:**
- Taiwan ordinary share: `2303.TW`, priced in TWD.
- US ADR: `UMC` (NYSE), priced in USD.
- ADR ratio: **1 ADR = 5 ordinary shares**.
- FX: `TWD=X` = USD/TWD (TWD per 1 USD).

So the ADR's implied price per ordinary share, in TWD, is:

    adr_implied_twd = UMC_usd / 5 * (TWD per USD)

If they are the same asset (no arb, ignoring frictions), `adr_implied_twd ≈ 2303.TW close`.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ADR_RATIO = 5  # 1 ADR = 5 ordinary shares

tickers = ["2303.TW", "UMC", "TWD=X"]
raw = yf.download(tickers, period="10y", auto_adjust=False)["Close"]
raw = raw.rename(columns={"2303.TW": "tw_twd", "UMC": "adr_usd", "TWD=X": "usdtwd"})
raw.tail()

In [ ]:
# Align on common trading days, forward-fill FX gaps, drop rows missing any leg.
df = raw.copy()
df["usdtwd"] = df["usdtwd"].ffill()
df = df.dropna(subset=["tw_twd", "adr_usd", "usdtwd"])

# ADR implied price per ordinary share, in TWD.
df["adr_implied_twd"] = df["adr_usd"] / ADR_RATIO * df["usdtwd"]

# Premium/discount of ADR vs local (%).
df["premium_pct"] = (df["adr_implied_twd"] / df["tw_twd"] - 1) * 100
df[["tw_twd", "adr_implied_twd", "premium_pct"]].tail()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 9), sharex=True,
                       gridspec_kw={"height_ratios": [3, 1]})

ax[0].plot(df.index, df["tw_twd"], label="2303.TW (local, TWD)", lw=1.1)
ax[0].plot(df.index, df["adr_implied_twd"], label="UMC ADR implied (TWD)",
           lw=1.1, alpha=0.8)
ax[0].set_ylabel("Price per ordinary share (TWD)")
ax[0].set_title("UMC: Taiwan ordinary vs ADR-implied (normalized by 1:5 ratio + FX)")
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].axhline(0, color="k", lw=0.6)
ax[1].plot(df.index, df["premium_pct"], color="C3", lw=0.8)
ax[1].set_ylabel("ADR premium (%)")
ax[1].set_xlabel("Date")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# How tightly do they track? Correlation of daily returns + premium stats.
ret = df[["tw_twd", "adr_implied_twd"]].pct_change().dropna()
corr = ret["tw_twd"].corr(ret["adr_implied_twd"])

print(f"Daily-return correlation: {corr:.4f}")
print(f"ADR premium (%)  mean {df['premium_pct'].mean():+.2f}  "
      f"std {df['premium_pct'].std():.2f}  "
      f"min {df['premium_pct'].min():+.2f}  max {df['premium_pct'].max():+.2f}")
print("\nInterpretation: high return-correlation + a small, mean-reverting")
print("premium band => effectively the same asset (premium = arb/FX/timing friction).")
print("Note: TW and US sessions don't overlap, so daily 'premium' partly reflects")
print("non-synchronous closes, not just true mispricing.")

---

# 0050 (元大台灣50 ETF) vs TXO underlying (TAIEX index)

Question: **Can I treat 0050 and the TXO index (TAIEX) as the same asset?**

Unlike the ADR case above (2303 vs UMC are literally the *same* shares, just
re-expressed via ratio + FX), here the two are **different baskets**:

- `TXO` settles on the **TAIEX** (`^TWII`) — the cap-weighted index of ~all
  listed Taiwan stocks.
- `0050` is an **ETF holding only the top ~50 names** (TSMC alone is ~50% of it).

They share the same large-cap beta, so they'll be highly correlated day-to-day.
The real test is whether their **normalized levels stay locked together** (like
the ADR premium, a small mean-reverting band) or **drift apart over time**
(composition/tracking difference = a directional basis, not an arbable one).


In [ ]:
# Download 0050 ETF and the TAIEX index (TXO underlying), align on common days.
etf_idx = yf.download(["0050.TW", "^TWII"], period="10y", auto_adjust=False)["Close"]
etf_idx = etf_idx.rename(columns={"0050.TW": "etf_0050", "^TWII": "taiex"})
ei = etf_idx.dropna(subset=["etf_0050", "taiex"]).copy()

# Normalize both to 100 at the first common day so different scales
# (ETF ~NT$170 vs index ~23,000 pts) are comparable.
ei["n_0050"] = ei["etf_0050"] / ei["etf_0050"].iloc[0] * 100
ei["n_taiex"] = ei["taiex"] / ei["taiex"].iloc[0] * 100

# "Premium" here = how far 0050 has drifted from the index, normalized-level basis.
ei["basis_pct"] = (ei["n_0050"] / ei["n_taiex"] - 1) * 100
ei[["n_0050", "n_taiex", "basis_pct"]].tail()

In [ ]:
# Top: normalized levels (do they move in unison?). Bottom: the drift/basis.
fig, ax = plt.subplots(2, 1, figsize=(13, 9), sharex=True,
                       gridspec_kw={"height_ratios": [3, 1]})

ax[0].plot(ei.index, ei["n_0050"], label="0050 ETF (normalized=100)", lw=1.1)
ax[0].plot(ei.index, ei["n_taiex"], label="TAIEX / TXO index (normalized=100)",
           lw=1.1, alpha=0.8)
ax[0].set_ylabel("Normalized level (start = 100)")
ax[0].set_title("0050 ETF vs TAIEX (TXO underlying) — normalized to a common start")
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].axhline(0, color="k", lw=0.6)
ax[1].plot(ei.index, ei["basis_pct"], color="C3", lw=0.8)
ax[1].set_ylabel("0050 − TAIEX drift (%)")
ax[1].set_xlabel("Date")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Quant test: daily-return correlation vs the ADR case, and is the basis
# mean-reverting (like a real parity) or trending (structural divergence)?
ret = ei[["etf_0050", "taiex"]].pct_change().dropna()
corr = ret["etf_0050"].corr(ret["taiex"])

basis = ei["basis_pct"]
# Trend strength: fraction of basis variance explained by a straight line in time.
t = np.arange(len(basis))
slope, intercept = np.polyfit(t, basis.values, 1)
fit = slope * t + intercept
r2_trend = 1 - np.var(basis.values - fit) / np.var(basis.values)

print(f"Daily-return correlation (0050 vs TAIEX): {corr:.4f}")
print(f"Drift/basis (%)   start {basis.iloc[0]:+.2f}   end {basis.iloc[-1]:+.2f}   "
      f"min {basis.min():+.2f}   max {basis.max():+.2f}")
print(f"Linear trend in basis: {slope*252:+.2f} %/yr, R^2 {r2_trend:.2f} "
      f"(high R^2 => directional drift, NOT mean-reverting)")
print()
print("Compare with the ADR (2303 vs UMC): premium mean ~ -0.4%, std ~2%, "
      "flat/mean-reverting -> same asset.")

## Interpretation — can I treat 0050 and the TXO index as the same asset?

**No.** They move together *short-term* but are **not the same asset**, and the difference is *directional*, not a mean-reverting parity.

What the data shows (10y, ~2,430 days):

- **Daily-return correlation ≈ 0.95** — very high. Both are dominated by the same
  large-cap Taiwan beta (TSMC + friends), so day-to-day they *look* like they move
  in unison. This is why 0050 works as a rough **hedge/proxy** for short-horizon
  index exposure.
- **But the normalized levels fan apart:** the 0050 − TAIEX basis drifts from 0%
  to **≈ +22%** over the window (≈ **+1.3 %/yr**, with a clear upward trend). 0050
  structurally **outperforms** the broad index because it's concentrated in the
  top ~50 (TSMC ~50% weight), which led the market. Also 0050 distributes
  dividends while `^TWII` is a price index — another slow wedge.
- **Contrast with the ADR (2303 vs UMC):** there the premium is a small band
  (mean ≈ −0.4%, std ≈ 2%) that **mean-reverts around 0** — the signature of the
  *same* asset. Here the basis **trends** and does not revert.

**Bottom line for trading:** treat 0050 ≈ TXO index only as a **beta proxy / hedge
over short horizons**. Do **not** treat them as the same asset for parity or basis
arb: the basis is a persistent, drifting composition premium, so a "sell rich /
buy cheap" pair between them is a bet on relative performance (TSMC-concentration
vs broad market), **not** a converging arbitrage. Unlike the ADR case, there's no
reason the gap closes by expiry — it has trended one way for a decade.
